# 60M TinyStoriesV2-GPT4 Pretrain (Colab Pro A100)

End-to-end pretrain of the 60M-parameter base model. Expect **~4-6 hours** on A100 with bfloat16.

**Before starting:**
- Runtime → Change runtime type → A100 GPU, High-RAM if available.
- Have ~15GB free in Google Drive for checkpoints + tokenized data.

**Output:** `checkpoints/base_60m/final.pt` saved to Google Drive.

## 1. Setup: clone, install, mount Drive

In [ ]:
# Mount Drive for persistent checkpoints (survives Colab disconnects).
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = '/content/drive/MyDrive/transformer-lm'
os.makedirs(DRIVE_ROOT, exist_ok=True)
os.makedirs(f'{DRIVE_ROOT}/checkpoints/base_60m', exist_ok=True)
os.makedirs(f'{DRIVE_ROOT}/logs', exist_ok=True)
os.makedirs(f'{DRIVE_ROOT}/data', exist_ok=True)

In [ ]:
# Clone the repo (replace USER with your GitHub username if forked).
%cd /content
!git clone https://github.com/Qwerty0826/char-level-transformer-lm.git || (cd char-level-transformer-lm && git pull)
%cd /content/char-level-transformer-lm
!pip install -e . -q
!pip install datasets -q

In [ ]:
# Symlink data and checkpoint dirs to Drive so nothing is lost on disconnect.
!rm -rf data checkpoints logs
!ln -sf {DRIVE_ROOT}/data        data
!ln -sf {DRIVE_ROOT}/checkpoints checkpoints
!ln -sf {DRIVE_ROOT}/logs        logs
!ls -la data checkpoints logs

## 2. Verify GPU and that the test suite passes

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM:', torch.cuda.get_device_properties(0).total_memory / 1e9, 'GB')
print('bf16 supported:', torch.cuda.is_bf16_supported())

!python -m pytest tests/test_padding_mask.py tests/test_advanced.py tests/test_training.py -q

## 3. Download + tokenize TinyStoriesV2-GPT4

Idempotent — re-running skips work already done. First run takes ~1 hour (download + BPE train + encode).

In [ ]:
!python scripts/download_tinystories_v2.py --output_dir data/

## 4. Pretrain 60M base model

Config: `configs/tinystories_60m.yaml`. Architecture: d_model=640, 10 layers, 10 heads, 2 kv heads (5:1 GQA), context 512, vocab 16K. Total ~60M params. Trains for 20K steps, bf16, torch.compile, lr_max=3e-4. **Checkpoints every 500 steps to Drive** — Colab disconnects are survivable.

**Resuming:** add `--resume checkpoints/base_60m/step_XXXXXX.pt` if the session disconnects.

In [ ]:
!python scripts/train.py \
    --config configs/tinystories_60m.yaml \
    --device cuda

## 5. Sanity check the trained model

In [ ]:
!python scripts/generate.py \
    --checkpoint checkpoints/base_60m/final.pt \
    --vocab data/tinystories_v2_vocab.json \
    --merges data/tinystories_v2_merges.txt \
    --prompt 'Once upon a time there was a' \
    --max_tokens 200 \
    --temperature 0.8 --top_p 0.95

## Sanity gate

Before moving on to SFT:

- [ ] **val PPL ≤ 4.5** by step ~20K. Inspect the training logs — `[val]` lines show val loss and PPL.
- [ ] Generated sample is coherent multi-paragraph English with sensible story structure.
- [ ] `checkpoints/base_60m/final.pt` is on Drive.

If val PPL > 5.0, **stop**. Lower `lr_max` to 1e-4 and retry. Do not move on to SFT until the base model is clean.